# #272 validation: BEIR non-regression gate on hybrid default

Issue #272 swapped sqlite-vec's default L2 metric for `distance_metric=cosine`
and rebuilt the `vec_chunks` table in-place on v1 DBs.  Locally the full
5-dataset BEIR regression takes ~1h45m on CPU; this notebook monkey-patches
`vstash.embed.embed_texts` / `embed_query` to route through a GPU-backed
SentenceTransformer so embedding runs on T4 CUDA with 256-sized batches.
Expected runtime on T4: **~5-10 min** for all five BEIR datasets.

Pass criterion: **every dataset within `REGRESSION_TOLERANCE = 0.005`
absolute NDCG@10 of baseline**, matching the pytest gate in
`tests/test_beir_regression.py`.

Model is **plain BGE-small-en-v1.5** (not the tuned v2/v3) because the
baseline was established on the unmodified pipeline.  Tuned-model
validation lives in `beir_benchmark_colab.ipynb`.

In [ ]:
# Cell 1: Setup.  Clone the #272 feature branch.
#
# Colab kernels sometimes inherit a stale cwd after a previous rm -rf,
# which breaks every subsequent shell magic.  Force the kernel back to
# a known-good directory before any `!` command runs.
import os

os.chdir("/")
os.chdir("/content")

!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0'
!rm -rf /content/vstash
!git clone --branch feature/272-vec0-cosine-metric https://github.com/stffns/vstash.git /content/vstash

os.chdir("/content/vstash")
!pip install -q -e .
print("cwd:", os.getcwd())

# Sanity check: the vec_chunks DDL in store.py must declare cosine.
# If this assert fires, we cloned the wrong branch or a stale cache.
with open("vstash/store.py") as f:
    src = f.read()
assert "distance_metric=cosine" in src, "branch does not contain the #272 fix"
assert 'SCHEMA_VERSION = "2"' in src, "branch is not at schema v2"
assert "distance_cutoff: float = 1.3225" in src, (
    "branch missing the 1.15 -> 1.3225 distance_cutoff fix"
)
assert "0.9, 0.1, 25.0" in src, "branch missing the 5.0 -> 25.0 long-query cutoff fix"
print("#272 markers present: OK")

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Monkey-patch vstash.embed with a GPU-backed SentenceTransformer
# BEFORE importing experiments.beir_benchmark (which does
# ``from vstash.embed import embed_texts, embed_query`` and captures the
# function refs at import time, so any patch after that import is a
# no-op).  Then run BEIR in-process so the patch actually applies.
#
# What this buys: the default ONNX backend is CPU-only.  On Colab T4,
# routing through a CUDA SentenceTransformer with batch_size=256 drops
# NFCorpus embed time from 378s to ~15s (measured 2026-04-24).
import os
import sys
import time
from pathlib import Path

os.chdir("/content/vstash")
sys.path.insert(0, ".")
os.makedirs("experiments/data", exist_ok=True)

stale = Path("experiments/results/beir_benchmark.json")
if stale.exists():
    stale.unlink()
    print(f"cleared stale results file: {stale}")

# 1. Load the SentenceTransformer once on GPU.
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256 if DEVICE == "cuda" else 64
print(f"GPU-accelerated embed: device={DEVICE}, batch_size={BATCH_SIZE}")

_MODEL_CACHE: dict[str, SentenceTransformer] = {}


def _get_model(model_name: str) -> SentenceTransformer:
    if model_name not in _MODEL_CACHE:
        t0 = time.perf_counter()
        _MODEL_CACHE[model_name] = SentenceTransformer(model_name, device=DEVICE)
        print(f"  loaded {model_name} on {DEVICE} in {time.perf_counter() - t0:.1f}s")
    return _MODEL_CACHE[model_name]


def gpu_embed_texts(texts, model_name, backend="auto"):
    if not texts:
        return []
    model = _get_model(model_name)
    vecs = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=DEVICE,
    )
    return [list(map(float, v)) for v in vecs]


def gpu_embed_query(text, model_name, backend="auto"):
    return gpu_embed_texts([text], model_name, backend)[0]


# 2. Replace the vstash.embed module-level attributes BEFORE
# experiments.beir_benchmark is imported.  The benchmark script does
# ``from vstash.embed import embed_texts, embed_query`` which captures
# a reference to whatever ``vstash.embed.embed_texts`` is *at that
# moment*, so ordering matters here.
import vstash.embed as _vstash_embed

_original_embed_texts = _vstash_embed.embed_texts
_original_embed_query = _vstash_embed.embed_query
_vstash_embed.embed_texts = gpu_embed_texts
_vstash_embed.embed_query = gpu_embed_query
print("monkey-patched vstash.embed.embed_texts / embed_query -> GPU path")

# 3. Run the BEIR benchmark in-process.  ``main()`` reads sys.argv.
sys.argv = [
    "experiments.beir_benchmark",
    "--no-chroma",
    "--model",
    "BAAI/bge-small-en-v1.5",
]
try:
    from experiments.beir_benchmark import main as _beir_main

    t0 = time.perf_counter()
    _beir_main()
    print(f"\ntotal wall time: {time.perf_counter() - t0:.1f}s")
finally:
    # Restore the originals so subsequent cells see the unmodified
    # module (Cell 4 runs pytest which spawns its own process and is
    # unaffected; this restore is belt-and-suspenders for interactive
    # re-runs of Cell 2).
    _vstash_embed.embed_texts = _original_embed_texts
    _vstash_embed.embed_query = _original_embed_query

In [ ]:
# Cell 3: Compare against the frozen v1 baseline and report pass/fail.
# Same criterion as tests/test_beir_regression.py (absolute NDCG@10
# within REGRESSION_TOLERANCE of baseline).
import json
from pathlib import Path

REGRESSION_TOLERANCE = 0.005

results_path = Path("experiments/results/beir_benchmark.json")
baseline_path = Path("experiments/results/beir_adaptive_baseline.json")
assert results_path.exists(), "beir_benchmark.json not found. Cell 2 must complete first."
assert baseline_path.exists(), "baseline JSON missing -- repo is incomplete."

results = json.loads(results_path.read_text())
baseline = json.loads(baseline_path.read_text())

assert results["model"] == "BAAI/bge-small-en-v1.5", (
    f"Cell 2 must be run on BGE-small to compare against the paper baseline, got {results['model']!r}"
)

print(f"Model:              {results['model']}")
print(f"Baseline timestamp: {baseline.get('timestamp', 'unknown')}")
print(f"Tolerance:          +/- {REGRESSION_TOLERANCE:.3f} absolute NDCG@10")
print()
print(f"{'Dataset':<10} {'Baseline':>10} {'Actual':>10} {'Delta':>10}  {'Verdict':<12}")
print("-" * 60)

failures: list[tuple[str, float, float]] = []
actual_by_dataset = {r["dataset"]: r["vstash"]["ndcg_10"] for r in results["results"]}

for dataset, bvals in baseline["results"].items():
    expected = bvals["ndcg_10"]
    actual = actual_by_dataset.get(dataset)
    if actual is None:
        print(f"{dataset:<10} {expected:>10.4f}     (skipped by Cell 2)")
        continue
    delta = actual - expected
    verdict = "PASS" if delta >= -REGRESSION_TOLERANCE else "FAIL"
    if verdict == "FAIL":
        failures.append((dataset, expected, actual))
    print(f"{dataset:<10} {expected:>10.4f} {actual:>10.4f} {delta:>+10.4f}  {verdict:<12}")

print()
if failures:
    print(f"REGRESSION: {len(failures)} dataset(s) below tolerance.")
    for ds, exp, act in failures:
        print(f"  {ds}: {act:.4f} < {exp:.4f} - {REGRESSION_TOLERANCE}")
    raise SystemExit(1)
else:
    print(
        "GATE PASS: all 5 BEIR datasets within tolerance.  #272 cosine migration is safe to merge."
    )